<a href="https://colab.research.google.com/github/LucasArtoni1983/orchid-log-analise-dados/blob/main/orchidLog_case.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importando bibliotecas

In [4]:
#manipulação e análise de datasets
import pandas as pd

# Etapa 1

In [5]:
#Armazenando as bases de dados em datasets separados
df_entregas_orquideas = pd.read_csv("/content/entregas_orquideas.csv")
df_estufas_qualidade = pd.read_csv("/content/estufas_qualidade.csv")
df_telemetria_iot = pd.read_csv("/content/telemetria_iot.csv")

In [6]:
#Histórico logístico de entregas
df_entregas_orquideas.head()

,ID_Carga,Placa_Caminhao,Estufa_Origem,UF_Destino,Qtd_Orquideas,Status_Entrega
0,LOG-1000,TX-103,EST-3,SP,457,Entregue
1,LOG-1001,TX-100,EST-1,SC,165,Entregue
2,LOG-1002,TX-108,EST-2,MG,124,Entregue
3,LOG-1003,TX-107,EST-1,RJ,162,Entregue
4,LOG-1004,TX-107,EST-2,SP,469,Entregue


In [7]:
#Variedades e responsáveis técnicos de orquídeas
df_estufas_qualidade.head()

,id_estufa,especie_predominante,umidade_media_pct,responsavel_tecnico,indice_qualidade_lote
0,EST-1,Phalaenopsis,75,Ana Silva,0.95
1,EST-2,Cattleya,82,Carlos Mendes,0.88
2,EST-3,Dendrobium,65,Mariana Costa,0.91


In [8]:
#Identificação dos veículos e temperaturas internas
df_telemetria_iot.head()

,Placa_Caminhao,Temperatura_C,Alerta_Calor
0,TX-103,20,NAO
1,TX-100,30,SIM
2,TX-108,28,SIM
3,TX-107,33,SIM
4,TX-104,20,NAO


# Etapa 2

In [9]:
# Temperaturas acima de 28ºC danificam as orquídeas de forma irreversível.
# Mesmo existindo uma coluna similar(Alerta_Calor),
# optei por criar uma nova coluna como exercício de estruturas de repetição

list_status_temperatura = []

list_temps = df_telemetria_iot['Temperatura_C'].tolist()

for temp in list_temps:
    if temp >= 28:
        list_status_temperatura.append("Alerta")
    else:
        list_status_temperatura.append("OK")

display(list_status_temperatura)

df_telemetria_iot['status_temperatura'] = list_status_temperatura

['OK',
 'Alerta',
 'Alerta',
 'Alerta',
 'OK',
 'OK',
 'Alerta',
 'Alerta',
 'Alerta',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'OK',
 'Alerta']

In [10]:
#exibição dos 5 primeiros registros
df_telemetria_iot.head()

,Placa_Caminhao,Temperatura_C,Alerta_Calor,status_temperatura
0,TX-103,20,NAO,OK
1,TX-100,30,SIM,Alerta
2,TX-108,28,SIM,Alerta
3,TX-107,33,SIM,Alerta
4,TX-104,20,NAO,OK


# Etapa 3

In [11]:
# Total de orquídeas devidamente entregues
tot_orquideas = df_entregas_orquideas['Qtd_Orquideas'].sum()

print(f"No total foram entregues {tot_orquideas} orquídeas")

No total foram entregues 55425 orquídeas


In [12]:
# Orquídeas perdidas

orq_perdidas = df_entregas_orquideas[df_entregas_orquideas['Status_Entrega'] == 'Cancelado']

soma_orq_perdidas = orq_perdidas['Qtd_Orquideas'].sum()

print(f'No total foram perdidas {soma_orq_perdidas} orquídeas')

No total foram perdidas 6349 orquídeas


In [13]:
# Estado com as maiores perdas absolutas de orquídeas
perdas_por_estado = orq_perdidas['UF_Destino'].value_counts().idxmax()
perdas_por_estado
print(f'O estado com mais perdas foi {perdas_por_estado}')

O estado com mais perdas foi SP


In [14]:
# Temperatura máxima e temperatura média
# registradas nos caminhões
temp_max = df_telemetria_iot['Temperatura_C'].max()
print(f'A temperatura máxima registrada foi de: {temp_max}ºC')

temp_media = df_telemetria_iot['Temperatura_C'].mean()
print(f'A temperatura máxima registrada foi de: {temp_media}ºC')

A temperatura máxima registrada foi de: 34ºC
A temperatura máxima registrada foi de: 27.0ºC


In [15]:
# A taxa de cancelamentos relativa é mais justa, devido ao volume de pedidos por estado

# Entregas canceladas agrupadas por estado
cancelados = df_entregas_orquideas[(df_entregas_orquideas['Status_Entrega'] == 'Cancelado') & (df_entregas_orquideas['UF_Destino'] == 'SP')]
cancelados
total_entregas = df_entregas_orquideas[df_entregas_orquideas['UF_Destino'] == 'SP']

In [16]:
#Taxa de cancelamentos aproximada em SP
taxa_SP = len(cancelados) / len(total_entregas)
taxa_SP

0.07777777777777778

In [17]:
# Criação de coluna booleana de cancelamentos
# para facilitar o agrupamento
df_entregas_orquideas['eh_cancelado'] = df_entregas_orquideas['Status_Entrega'] == 'Cancelado'

In [22]:
# Agrupamento de entregas por estado
df_gb_estados = df_entregas_orquideas.groupby('UF_Destino')
df_gb_estados.size()

,0
UF_Destino,
MG,27
PR,32
RJ,32
SC,19
SP,90


In [23]:
# Entregas canceladas por estado
cancelados_por_estado = df_gb_estados['eh_cancelado'].sum()
cancelados_por_estado

,eh_cancelado
UF_Destino,
MG,6
PR,6
RJ,5
SC,3
SP,7


In [24]:
# Taxa aproximada de cancelamentos por estado
taxa_cancelamento_por_estado = (cancelados_por_estado / df_gb_estados.size() * 100).round(1)
taxa_cancelamento_por_estado

,0
UF_Destino,
MG,22.2
PR,18.8
RJ,15.6
SC,15.8
SP,7.8
